# 🛍️ Mall Customer Segmentation using K-Means Clustering

**Author:** Kailash  
**Dataset:** Mall Customers Dataset (200 records)  
**Objective:** Identify distinct customer groups based on Annual Income and Spending Score to help the marketing team design targeted campaigns.

---

## 📌 Project Overview

Customer segmentation is the practice of dividing customers into groups that share similar characteristics. In this project, we apply **K-Means Clustering** — an unsupervised machine learning algorithm — to segment mall customers based on their:

- **Annual Income (k\$)**
- **Spending Score (1–100)**

The result helps businesses identify high-value customers, budget shoppers, and everything in between.

## 📂 Table of Contents
1. [Import Libraries](#1)
2. [Load & Explore Data](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Feature Selection](#4)
5. [Finding Optimal Clusters – Elbow Method](#5)
6. [Training the K-Means Model](#6)
7. [Visualizing Customer Segments](#7)
8. [Business Insights](#8)

---
## 1. Import Libraries <a id='1'></a>

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Settings
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print("✅ Libraries imported successfully!")

---
## 2. Load & Explore Data <a id='2'></a>

In [ ]:
# Load dataset
df = pd.read_csv('Mall_Customers.csv')

print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}\n")
df.head(10)

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing Values per Column:")
print(missing)
print(f"\n✅ No missing values found!" if missing.sum() == 0 else "⚠️ Missing values detected!")

In [ ]:
# Statistical summary
df.describe().round(2)

---
## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

> Understanding the distribution and relationships in the data before modeling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender distribution
gender_counts = df['Gender'].value_counts()
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#FF6B6B', '#4ECDC4'], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Gender Distribution', fontsize=14, fontweight='bold')

# Age distribution
axes[1].hist(df['Age'], bins=20, color='#6C63FF', edgecolor='white', alpha=0.85)
axes[1].set_title('Age Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')

plt.suptitle('Customer Demographics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual income distribution
axes[0].hist(df['Annual Income (k$)'], bins=20, color='#F7B731', edgecolor='white', alpha=0.85)
axes[0].set_title('Annual Income Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Count')

# Spending score distribution
axes[1].hist(df['Spending Score (1-100)'], bins=20, color='#20BF6B', edgecolor='white', alpha=0.85)
axes[1].set_title('Spending Score Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Spending Score (1-100)')
axes[1].set_ylabel('Count')

plt.suptitle('Income & Spending Patterns', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
numeric_df = df.select_dtypes(include='number').drop('CustomerID', axis=1)
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Annual Income vs Spending Score
plt.figure(figsize=(9, 6))
colors = {'Male': '#4ECDC4', 'Female': '#FF6B6B'}
for gender, group in df.groupby('Gender'):
    plt.scatter(group['Annual Income (k$)'], group['Spending Score (1-100)'],
                c=colors[gender], label=gender, alpha=0.75, edgecolors='white', s=60)
plt.title('Annual Income vs Spending Score (by Gender)', fontsize=14, fontweight='bold')
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Feature Selection <a id='4'></a>

We use **Annual Income** and **Spending Score** as our clustering features — these two variables capture the key behavioral and economic traits of each customer.

In [ ]:
# Select features for clustering
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

print(f"Feature matrix shape: {X.shape}")
print(f"Sample (first 5 rows):\n{X[:5]}")

---
## 5. Finding Optimal Clusters – Elbow Method <a id='5'></a>

The **Elbow Method** plots the Within-Cluster Sum of Squares (WCSS) for different values of `k`. The optimal number of clusters is where the curve bends sharply — like an elbow.

In [ ]:
# Calculate WCSS for k = 1 to 10
wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', max_iter=300, random_state=42)
    km.fit(X)
    wcss.append(km.inertia_)

# Plot the Elbow Curve
plt.figure(figsize=(9, 5))
plt.plot(K_range, wcss, 'bo-', linewidth=2, markersize=8, color='#6C63FF')
plt.axvline(x=5, color='red', linestyle='--', linewidth=1.5, label='Optimal k = 5')
plt.title('Elbow Method – Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('WCSS (Inertia)', fontsize=12)
plt.legend(fontsize=11)
plt.xticks(K_range)
plt.tight_layout()
plt.show()

print("📌 The elbow appears at k=5 — this is our optimal number of clusters.")

---
## 6. Training the K-Means Model <a id='6'></a>

In [ ]:
# Train K-Means with k=5
kmeans = KMeans(n_clusters=5, init='k-means++', max_iter=300, random_state=42)
y_pred = kmeans.fit_predict(X)

# Add cluster labels to original dataframe
df['Cluster'] = y_pred

print(f"Cluster Centers (Annual Income, Spending Score):")
for i, center in enumerate(kmeans.cluster_centers_):
    print(f"  Cluster {i+1}: Income = {center[0]:.1f}k$, Spending Score = {center[1]:.1f}")

In [ ]:
# Cluster-wise summary statistics
cluster_summary = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean().round(2)
cluster_summary['Count'] = df.groupby('Cluster')['CustomerID'].count()
cluster_summary.index = [f'Cluster {i+1}' for i in cluster_summary.index]
print("Cluster Summary Statistics:")
cluster_summary

---
## 7. Visualizing Customer Segments <a id='7'></a>

In [ ]:
# Color palette for clusters
colors = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#6C63FF', '#20BF6B']
segment_labels = [
    'Low Income, Low Spend',
    'High Income, Low Spend',
    'Moderate Income & Spend',
    'Low Income, High Spend',
    'High Income, High Spend'
]

plt.figure(figsize=(11, 8))

for i in range(5):
    plt.scatter(
        X[y_pred == i, 0],
        X[y_pred == i, 1],
        s=70,
        c=colors[i],
        label=f'Cluster {i+1}: {segment_labels[i]}',
        edgecolors='white',
        linewidth=0.5,
        alpha=0.9
    )

# Plot centroids
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    s=250,
    c='black',
    marker='X',
    label='Centroids',
    zorder=5
)

plt.title('🛍️ Mall Customer Segments (K-Means, k=5)', fontsize=15, fontweight='bold')
plt.xlabel('Annual Income (k$)', fontsize=13)
plt.ylabel('Spending Score (1–100)', fontsize=13)
plt.legend(loc='upper left', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig('customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved as 'customer_segments.png'")

---
## 8. Business Insights <a id='8'></a>

After segmenting 200 mall customers into **5 distinct groups**, here are the key findings:

| Cluster | Label | Annual Income | Spending Score | Business Action |
|---------|-------|--------------|----------------|------------------|
| 1 | Careful Shoppers | Low (~26k\$) | Low (~20) | Budget deals & discount campaigns |
| 2 | Conservative Spenders | High (~87k\$) | Low (~18) | Premium product upselling |
| 3 | Standard Customers | Mid (~56k\$) | Mid (~50) | Loyalty rewards program |
| 4 | Impulsive Buyers | Low (~26k\$) | High (~79) | Flash sales & limited offers |
| 5 | ⭐ Target Customers | High (~87k\$) | High (~82) | VIP/Premium membership programs |

### 🎯 Key Takeaways
- **Cluster 5 (High Income, High Spend)** is the most valuable segment — they earn well and spend freely.
- **Cluster 4 (Low Income, High Spend)** are impulsive buyers — great targets for flash sales.
- **Cluster 2 (High Income, Low Spend)** has untapped potential — premium product recommendations could convert them.
- **Cluster 1 (Low Income, Low Spend)** may respond best to discount-driven campaigns.

In [ ]:
# Cluster size bar chart
cluster_counts = df['Cluster'].value_counts().sort_index()
labels = [f'C{i+1}' for i in cluster_counts.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, cluster_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, cluster_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', fontweight='bold', fontsize=12)
plt.title('Number of Customers per Cluster', fontsize=14, fontweight='bold')
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Customer Count', fontsize=12)
plt.tight_layout()
plt.show()